In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(r"dataset_2.csv")

ModuleNotFoundError: No module named 'pandas'

In [7]:
print(df['BMI'].mean())

NameError: name 'df' is not defined

In [10]:
print(df['BMI'].median())

21.9


In [11]:
df.groupby('Gender')['BMI'].mean()

Gender
0    21.625000
1    22.559091
Name: BMI, dtype: float64

In [12]:
df.groupby('Gender')['BMI'].median()

Gender
0    21.75
1    22.40
Name: BMI, dtype: float64

In [13]:
df.pivot_table(values='BMI', index='Gender', aggfunc='mean')

,BMI
Gender,
0,21.625000
1,22.559091


In [14]:
# df.shape        # (rows, columns)
# df.dtypes       # column types
df.describe()   # summary stats for all numeric columns
# df.isnull().sum() # count missing values per column

,ID,Gender,DailyScreenTimeHours,WeeklyExerciseHours,FastFoodMealsPerWeek,BMI
count,50.00000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,25.50000,0.440000,4.362000,3.018000,2.500000,22.036000
std,14.57738,0.501427,1.483637,1.246332,1.821078,1.906997
min,1.00000,0.000000,1.000000,0.000000,0.000000,17.600000
25%,13.25000,0.000000,3.275000,2.000000,1.000000,21.150000
50%,25.50000,0.000000,4.650000,3.200000,2.500000,21.900000
75%,37.75000,1.000000,5.400000,3.875000,4.000000,23.200000
max,50.00000,1.000000,6.600000,5.500000,5.000000,27.200000


In [15]:
# Calculate BMI interquartile range (IQR)
q1 = df['BMI'].quantile(0.25)
q3 = df['BMI'].quantile(0.75)
iqr = q3 - q1
print(f"BMI IQR: {iqr:.2f} (Q1={q1:.2f}, Q3={q3:.2f})")

# IQR by gender
bmi_iqr_by_gender = df.groupby('Gender')['BMI'].agg(lambda x: x.quantile(0.75) - x.quantile(0.25))
print('\nBMI IQR by Gender:')
print(bmi_iqr_by_gender)


BMI IQR: 2.05 (Q1=21.15, Q3=23.20)

BMI IQR by Gender:
Gender
0    1.625
1    2.550
Name: BMI, dtype: float64


In [16]:
# Calculate IQR for Daily Screen Time and Weekly Exercise
for col, label in [
    ('DailyScreenTimeHours', 'Daily Screen Time (hours)'),
    ('WeeklyExerciseHours', 'Weekly Exercise (hours)')
]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    print(f"{label} IQR: {iqr:.2f} (Q1={q1:.2f}, Q3={q3:.2f})")

    iqr_by_gender = df.groupby('Gender')[col].agg(lambda x: x.quantile(0.75) - x.quantile(0.25))
    print(f"\n{label} IQR by Gender:")
    print(iqr_by_gender)
    print('\n' + '-' * 40 + '\n')


Daily Screen Time (hours) IQR: 2.12 (Q1=3.28, Q3=5.40)

Daily Screen Time (hours) IQR by Gender:
Gender
0    2.150
1    1.675
Name: DailyScreenTimeHours, dtype: float64

----------------------------------------

Weekly Exercise (hours) IQR: 1.88 (Q1=2.00, Q3=3.88)

Weekly Exercise (hours) IQR by Gender:
Gender
0    1.925
1    1.800
Name: WeeklyExerciseHours, dtype: float64

----------------------------------------



In [17]:
print(df['WeeklyExerciseHours'].median())

3.2


In [18]:
print(df['FastFoodMealsPerWeek'].median())

2.5


In [19]:
# BMI standard deviation by gender
print("BMI Standard Deviation by Gender:")
print(df.groupby('Gender')['BMI'].std())


BMI Standard Deviation by Gender:
Gender
0    1.895536
1    1.831382
Name: BMI, dtype: float64


In [4]:
from scipy.stats import pearsonr, linregress

pairs = [
    ('FastFoodMealsPerWeek', 'Fast food meals per week'),
    ('DailyScreenTimeHours', 'Daily Screen Time (hours)'),
    ('WeeklyExerciseHours', 'Weekly Exercise (hours)'),
]

print('Gendered regression results between BMI and predictors:')
for gender, subset in df.groupby('Gender'):
    print(f"\nGender: {gender}")
    for col, label in pairs:
        clean = subset[[col, 'BMI']].dropna()
        if len(clean) < 2:
            print(f"  {label}: insufficient data")
            continue
        reg = linregress(clean[col], clean['BMI'])
        print(f"  {label}: slope={reg.slope:.3f}, intercept={reg.intercept:.3f}, r={reg.rvalue:.3f}, r^2={reg.rvalue**2:.3f}, p={reg.pvalue:.3f}, stderr={reg.stderr:.3f}")


Gendered regression results between BMI and predictors:

Gender: 0
  Fast food meals per week: slope=0.640, intercept=20.001, r=0.555, r^2=0.308, p=0.002, stderr=0.188
  Daily Screen Time (hours): slope=0.421, intercept=19.894, r=0.330, r^2=0.109, p=0.086, stderr=0.236
  Weekly Exercise (hours): slope=-0.841, intercept=24.273, r=-0.559, r^2=0.312, p=0.002, stderr=0.245

Gender: 1
  Fast food meals per week: slope=0.462, intercept=21.426, r=0.520, r^2=0.271, p=0.013, stderr=0.169
  Daily Screen Time (hours): slope=-0.026, intercept=22.682, r=-0.021, r^2=0.000, p=0.927, stderr=0.283
  Weekly Exercise (hours): slope=0.077, intercept=22.341, r=0.052, r^2=0.003, p=0.819, stderr=0.330


In [5]:
from scipy.stats import pearsonr, linregress

pairs = [
    ('FastFoodMealsPerWeek', 'Fast food meals per week'),
    ('DailyScreenTimeHours', 'Daily Screen Time (hours)'),
    ('WeeklyExerciseHours', 'Weekly Exercise (hours)'),
]

print('Overall regression results between BMI and predictors:')
for col, label in pairs:
    clean = df[[col, 'BMI']].dropna()
    if len(clean) < 2:
        print(f"{label}: insufficient data")
        continue
    reg = linregress(clean[col], clean['BMI'])
    print(f"{label}: slope={reg.slope:.3f}, intercept={reg.intercept:.3f}, r={reg.rvalue:.3f}, r^2={reg.rvalue**2:.3f}, p={reg.pvalue:.3f}, stderr={reg.stderr:.3f}")


Overall regression results between BMI and predictors:
Fast food meals per week: slope=0.536, intercept=20.696, r=0.512, r^2=0.262, p=0.000, stderr=0.130
Daily Screen Time (hours): slope=0.283, intercept=20.803, r=0.220, r^2=0.048, p=0.125, stderr=0.181
Weekly Exercise (hours): slope=-0.485, intercept=23.500, r=-0.317, r^2=0.100, p=0.025, stderr=0.209
